## Configuration


In [ ]:
dbutils.widgets.text("catalog", "training_lab")
dbutils.widgets.text("schema", "student_01")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", schema)
ns = f"{catalog}.{schema}"
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")
print("Working namespace:", ns)
from pyspark.sql import functions as F


## One to many join


In [ ]:
display(spark.sql('SELECT o.order_id, o.amount, p.payment_id, p.paid\nFROM day18_orders o\nLEFT JOIN day18_payments p ON o.order_id = p.order_id\nORDER BY o.order_id, p.payment_id'))

## Preaggregate payments


In [ ]:
display(spark.sql('WITH payment_totals AS (\n SELECT order_id, SUM(paid) AS paid\n FROM day18_payments GROUP BY order_id\n)\nSELECT o.order_id, c.name, o.amount,\n       COALESCE(p.paid, 0) AS paid,\n       o.amount - COALESCE(p.paid, 0) AS outstanding\nFROM day18_orders o\nJOIN day18_customers c ON c.customer_id = o.customer_id\nLEFT JOIN payment_totals p ON p.order_id = o.order_id\nORDER BY o.order_id'))

## Scalar subquery


In [ ]:
display(spark.sql('SELECT order_id, amount FROM day18_orders\nWHERE amount > (SELECT AVG(amount) FROM day18_orders)\nORDER BY order_id'))

## Customers without orders


In [ ]:
display(spark.sql('SELECT c.customer_id, c.name FROM day18_customers c\nWHERE NOT EXISTS (\n SELECT 1 FROM day18_orders o\n WHERE o.customer_id = c.customer_id\n)'))

## Chained CTEs


In [ ]:
display(spark.sql('WITH customer_totals AS (\n SELECT customer_id, SUM(amount) AS revenue\n FROM day18_orders GROUP BY customer_id\n), ranked AS (\n SELECT *, DENSE_RANK() OVER (ORDER BY revenue DESC) AS rnk\n FROM customer_totals\n)\nSELECT c.name, r.revenue, r.rnk\nFROM ranked r JOIN day18_customers c\nON r.customer_id = c.customer_id\nWHERE r.rnk <= 2 ORDER BY r.rnk'))

## Windows


In [ ]:
display(spark.sql('SELECT order_id, customer_id, amount,\n ROW_NUMBER() OVER (\n  PARTITION BY customer_id ORDER BY amount DESC, order_id) AS rn,\n LAG(amount) OVER (\n  PARTITION BY customer_id ORDER BY order_date, order_id) AS previous,\n SUM(amount) OVER (\n  ORDER BY order_date, order_id\n  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running\nFROM day18_orders\nORDER BY order_date, order_id'))

## Moving average


In [ ]:
display(spark.sql('WITH daily AS (\n SELECT order_date, SUM(amount) AS revenue\n FROM day18_orders GROUP BY order_date\n)\nSELECT order_date, revenue,\n AVG(revenue) OVER (ORDER BY order_date\n  ROWS BETWEEN 1 PRECEDING AND CURRENT ROW) AS moving_2_rows\nFROM daily ORDER BY order_date'))